# 📊 Comparação das Três Soluções — MNIST

Notebook **autossuficiente**: treina as três arquiteturas (**LeNet**, **CNN Moderna** e **MLP**) por 10 épocas e compara as matrizes de confusão lado a lado. É só executar todas as células (no Colab, *Ambiente de execução → Executar tudo*).

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Felipe-Alcantara/Trabalho-2---Deep-Learning-aplicado-ao-MNist/blob/main/4_Comparacao.ipynb)

> 💡 **Dica:** rode no Google Colab com GPU (*Ambiente de execução → Alterar o tipo → GPU*) para treinar em poucos minutos.

## 0. Setup do ambiente

In [ ]:
# Setup — funciona no Google Colab (recomendado, com GPU) ou local.
# No Colab: Ambiente de execução > Alterar tipo > Acelerador de hardware: GPU.
import importlib.util, subprocess, sys

for pacote in ['seaborn', 'tensorflow']:
    if importlib.util.find_spec(pacote) is None:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pacote])

import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TensorFlow', tf.__version__, '| GPU disponível:', bool(gpus), gpus or '(rodando em CPU)')

## 1. Importações e dados

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from keras import utils as utls
from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import (Conv2D, MaxPooling2D, Activation, BatchNormalization,
                          Dropout, Flatten, Dense)
from sklearn.metrics import confusion_matrix

tf.random.set_seed(42); np.random.seed(42)

numClasses, batchSize, epochs = 10, 256, 10
(XTrain, yTrain), (XTest, yTest) = mnist.load_data()
XTrain = XTrain.astype('float32') / 255.0
XTest = XTest.astype('float32') / 255.0
yTestLabels = yTest.copy()
yTrainCat = utls.to_categorical(yTrain, numClasses)
yTestCat = utls.to_categorical(yTest, numClasses)

# Versões 2D (CNNs) e achatada (MLP)
XTrain2D, XTest2D = XTrain.reshape(-1, 28, 28, 1), XTest.reshape(-1, 28, 28, 1)
XTrainFlat, XTestFlat = XTrain.reshape(-1, 784), XTest.reshape(-1, 784)

## 2. Definição das três arquiteturas

In [ ]:
def criar_lenet():
    m = Sequential(name='LeNet')
    m.add(Conv2D(20, (5, 5), padding='same', input_shape=(28, 28, 1))); m.add(Activation('relu'))
    m.add(MaxPooling2D((2, 2), strides=2))
    m.add(Conv2D(50, (5, 5), padding='same')); m.add(Activation('relu'))
    m.add(MaxPooling2D((2, 2), strides=2))
    m.add(Flatten()); m.add(Dense(500)); m.add(Activation('relu'))
    m.add(Dense(numClasses)); m.add(Activation('softmax'))
    return m

def criar_cnn_moderna():
    m = Sequential(name='CNN_Moderna')
    m.add(Conv2D(32, (3, 3), padding='same', activation='relu', input_shape=(28, 28, 1)))
    m.add(BatchNormalization())
    m.add(Conv2D(32, (3, 3), padding='same', activation='relu')); m.add(BatchNormalization())
    m.add(MaxPooling2D((2, 2))); m.add(Dropout(0.25))
    m.add(Conv2D(64, (3, 3), padding='same', activation='relu')); m.add(BatchNormalization())
    m.add(Conv2D(64, (3, 3), padding='same', activation='relu')); m.add(BatchNormalization())
    m.add(MaxPooling2D((2, 2))); m.add(Dropout(0.25))
    m.add(Flatten()); m.add(Dense(256, activation='relu')); m.add(BatchNormalization())
    m.add(Dropout(0.5)); m.add(Dense(numClasses, activation='softmax'))
    return m

def criar_mlp():
    m = Sequential(name='MLP')
    m.add(Dense(512, activation='relu', input_shape=(784,))); m.add(Dropout(0.2))
    m.add(Dense(256, activation='relu')); m.add(Dropout(0.2))
    m.add(Dense(numClasses, activation='softmax'))
    return m

## 3. Treino das três redes (10 épocas cada)

In [ ]:
configs = [
    ('LeNet',       criar_lenet(),       XTrain2D,   XTest2D),
    ('CNN_Moderna', criar_cnn_moderna(), XTrain2D,   XTest2D),
    ('MLP',         criar_mlp(),         XTrainFlat, XTestFlat),
]

resultados = []
for nome, modelo, Xtr, Xte in configs:
    print(f'\n===== Treinando {nome} =====')
    modelo.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    hist = modelo.fit(Xtr, yTrainCat, batch_size=batchSize, epochs=epochs,
                      validation_data=(Xte, yTestCat), verbose=2)
    yPred = np.argmax(modelo.predict(Xte, verbose=0), axis=1)
    cm = confusion_matrix(yTestLabels, yPred)
    acc = float((yPred == yTestLabels).mean())
    resultados.append({'nome': nome, 'acc': acc, 'params': modelo.count_params(),
                       'cm': cm, 'val': hist.history['val_accuracy']})
    print(f'{nome}: acurácia de teste = {acc*100:.2f}%')

## 4. Tabela-resumo

In [ ]:
print(f"{'Arquitetura':<14} {'Parâmetros':>12} {'Acurácia':>10} {'Erros':>7}")
for r in resultados:
    erros = int(r['cm'].sum() - np.trace(r['cm']))
    print(f"{r['nome']:<14} {r['params']:>12,} {r['acc']*100:>9.2f}% {erros:>7}")

## 5. Matrizes de confusão lado a lado

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 6))
for ax, r in zip(axes, resultados):
    sns.heatmap(r['cm'], annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax, square=True)
    ax.set_title(f"{r['nome']}\nAcurácia: {r['acc']*100:.2f}%")
    ax.set_xlabel('Predito'); ax.set_ylabel('Real')
fig.suptitle('Matrizes de Confusão - MNIST (10 épocas)', fontsize=16, y=1.03)
plt.tight_layout(); plt.show()

## 6. Acurácia de validação por época

In [ ]:
plt.figure(figsize=(9, 5))
for r in resultados:
    plt.plot(range(1, len(r['val']) + 1), r['val'], 'o-', label=r['nome'])
plt.title('Acurácia de Validação por Época')
plt.xlabel('Época'); plt.ylabel('Acurácia (validação)')
plt.legend(); plt.grid(True, alpha=0.3); plt.show()

## 7. Conclusão

_Veja a análise textual completa no `README.md` da raiz do projeto (seção **Resultados e Conclusão**)._